# Project 8


I have used ChatGPT and Github Copilot for guidence and faster development

In [ ]:
!%pip install networkx matplotlib
!%pip install geopandas
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import zipfile
import requests
import tempfile
import os
import numpy as np
import requests
from io import BytesIO
import numpy as np
import matplotlib.pyplot as plt
import random
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import random
import seaborn as sns
from scipy import stats
import networkit as nk
import powerlaw
import collections
from scipy.stats import ks_2samp
from scipy.stats import poisson
import matplotlib.lines as mlines
from shapely.geometry import LineString
from pyproj import CRS, Transformer
import contextily as ctx
import heapq


P8.1 Prepare a code which draw for given network plot of the relation C(k)

In [ ]:
def plot_clustering_vs_degree(graph):

    clustering_coeffs = nx.clustering(graph)
    degrees = dict(graph.degree())
    degree_clustering = {}
    for node, degree in degrees.items():
        if degree not in degree_clustering:
            degree_clustering[degree] = []
        degree_clustering[degree].append(clustering_coeffs[node])
        
    avg_clustering_by_degree = {k: np.mean(v) for k, v in degree_clustering.items()}
    
    degrees_sorted = sorted(avg_clustering_by_degree.keys())
    clustering_values = [avg_clustering_by_degree[k] for k in degrees_sorted]
    
    plt.figure(figsize=(8, 6))
    plt.scatter(degrees_sorted, clustering_values, s=50, alpha=0.7, edgecolor='k')
    plt.title("Clustering coefficient vs degree", fontsize=14)
    plt.xlabel("Degree (k)", fontsize=12)
    plt.ylabel("Average clustering coefficient (C(k))", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()


G_hierarchical = nx.barabasi_albert_graph(n=1000, m=2)

plot_clustering_vs_degree(G_hierarchical)



P8.2 Find examples of hierarchical and non-hierarchical networks in available repositories, check by drawing graphs from the task P8.1.

In [ ]:
G_non_hierarchical = nx.erdos_renyi_graph(n=1000, p=0.01)
plot_clustering_vs_degree(G_non_hierarchical)

In [ ]:



agency = pd.read_csv('gtfs/agency.txt', sep=',', header=0)
calendar_dates = pd.read_csv('gtfs/calendar_dates.txt', sep=',', header=0)
calendar = pd.read_csv('gtfs/calendar.txt', sep=',', header=0)
#feed_info = pd.read_csv('gtfs/feed_info.txt', sep=',', header=0)
routes = pd.read_csv('gtfs/routes.txt', sep=',', header=0)
shapes = pd.read_csv('gtfs/shapes.txt', sep=',', header=0,)
stop_times = pd.read_csv('gtfs/stop_times.txt', sep=',', header=0)
stops = pd.read_csv('gtfs/stops.txt', sep=',', header=0)
trips = pd.read_csv('gtfs/trips.txt', sep=',', header=0)

# print(routes.head())
# print(shapes.head())
# print(stop_times.head())
# print(stops.head())
# print(trips.head())
 




In [ ]:
G = nx.Graph()

# Add nodes (stops) to the graph
for id, row in stops.iterrows():
    G.add_node(row["stop_id"], pos=(row['stop_lon'], row['stop_lat']), name=row['stop_name'])

# nx.draw(G, with_labels=True, node_color='lightblue', edge_color='gray', node_size=2000, font_size=15)

# plt.title("NetworkX Graph Example")
# plt.show()
stop_times_trips = pd.merge(stop_times, trips, on='trip_id')
stop_times_trips_routes = pd.merge(stop_times_trips, routes, on='route_id')
stop_times_trips_routes = stop_times_trips_routes[
    stop_times_trips_routes['stop_id'].isin(stops['stop_id'])
]
transport_types = {
    0: 'Tram', # Available in the dataset
    1: 'Metro',
    2: 'Rail', # Available in the dataset
    3: 'Bus', # Available in the dataset
    4: 'Ferry',
    5: 'Cable Car',
    6: 'Gondola',
    7: 'Funicular'
}
for route_type, transport_name in transport_types.items():
    
    transport_data = stop_times_trips_routes[stop_times_trips_routes['route_type'] == route_type]
    transport_data = transport_data.sort_values(['trip_id', 'stop_sequence'])
    
    for trip_id, trip_data in transport_data.groupby('trip_id'):
        
        stops_in_trip = trip_data['stop_id'].tolist()
        
        for i in range(len(stops_in_trip) - 1):
            from_stop = stops_in_trip[i]
            to_stop = stops_in_trip[i + 1]
            if from_stop in G and to_stop in G:
                G.add_edge(from_stop, to_stop, transport_type=transport_name)
            else: 
                # some stops are missing in the stops dataset
                print(f"Missing stops: {from_stop} -> {to_stop}")
                pass
                
pos = {node: data['pos'] for node, data in G.nodes(data=True)}



In [ ]:


transport_colors = {
    'Tram': 'orange',
    'Metro': 'red',
    'Rail': 'green',
    'Bus': 'blue',
    'Ferry': 'cyan',
    'Cable Car': 'purple',
    'Gondola': 'brown',
    'Funicular': 'pink'
}

edge_colors = []
for _, _, data in G.edges(data=True):
    transport_type = data.get('transport_type', 'Other')
    edge_colors.append(transport_colors.get(transport_type, 'gray'))

plt.figure(figsize=(12, 8))
#nx.draw_networkx_nodes(G, pos, node_size=10, node_color='black')
nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=1)

legend_elements = []
for transport_type, color in transport_colors.items():
    legend_elements.append(mlines.Line2D([], [], color=color, label=transport_type))
plt.legend(handles=legend_elements, title='Transport Types')

plt.axis('off')
plt.show()


In [ ]:
# Create a GeoDataFrame for nodes
nodes_gdf = gpd.GeoDataFrame(
    [(node, Point(data['pos'])) for node, data in G.nodes(data=True)],
    columns=['stop_id', 'geometry']
)

# Create a GeoDataFrame for edges
edges = []
for from_node, to_node, data in G.edges(data=True):
    from_pos = G.nodes[from_node]['pos']
    to_pos = G.nodes[to_node]['pos']
    line = LineString([from_pos, to_pos])  # Corrected line

    edges.append({'geometry': line, 'transport_type': data['transport_type']})

edges_gdf = gpd.GeoDataFrame(edges)

# Plotting
ax = nodes_gdf.plot(marker='o', color='black', markersize=5, figsize=(12, 8))
edges_gdf.plot(ax=ax, linewidth=1, column='transport_type', legend=True)
plt.show()


### Calculate the basic characteristics of each layer, and the whole network.


In [173]:
# Get the list of transport types present in the graph
transport_types_present = set(nx.get_edge_attributes(G, 'transport_type').values())

# Dictionary to hold subgraphs for each transport type
transport_subgraphs = {}

# Extract subgraphs for each transport type
for transport_type in transport_types_present:
    # Get edges of the current transport type
    edges = [(u, v) for u, v, d in G.edges(data=True) if d['transport_type'] == transport_type]
    # Create a subgraph
    G_sub = nx.Graph()
    G_sub.add_nodes_from(G.nodes(data=True))
    G_sub.add_edges_from(edges)
    transport_subgraphs[transport_type] = G_sub


In [171]:
def calculate_network_characteristics(G):
    characteristics = {}
    characteristics['Number of Nodes'] = G.number_of_nodes()
    characteristics['Number of Edges'] = G.number_of_edges()

    degrees = [deg for node, deg in G.degree()]
    characteristics['Average Degree'] = sum(degrees) / len(degrees) if degrees else 0

    # Network Density
    if G.number_of_nodes() > 1:
        characteristics['Density'] = nx.density(G)
    else:
        characteristics['Density'] = 0

    # Average Clustering Coefficient
    characteristics['Average Clustering Coefficient'] = nx.average_clustering(G)

    # Connected Components
    components = [len(c) for c in nx.connected_components(G)]
    characteristics['Number of Connected Components'] = len(components)
    characteristics['Largest Component Size'] = max(components) if components else 0

    # Subgraph of the largest connected component
    if components:
        largest_cc = max(nx.connected_components(G), key=len)
        G_lcc = G.subgraph(largest_cc)

        # Average Shortest Path Length
        if G_lcc.number_of_nodes() > 1:
            characteristics['Average Shortest Path Length'] = nx.average_shortest_path_length(G_lcc)
            # Diameter
            characteristics['Diameter'] = nx.diameter(G_lcc)
        else:
            characteristics['Average Shortest Path Length'] = 0
            characteristics['Diameter'] = 0
    else:
        characteristics['Average Shortest Path Length'] = 0
        characteristics['Diameter'] = 0

    return characteristics


In [ ]:
calculate_network_characteristics(G)

In [ ]:
# Calculate characteristics for each layer
layer_characteristics = {}
for transport_type, G_sub in transport_subgraphs.items():
    print(f"Calculating characteristics for {transport_type}...")
    characteristics = calculate_network_characteristics(G_sub)
    layer_characteristics[transport_type] = characteristics

# Calculate characteristics for the whole network
print("Calculating characteristics for the whole network...")
whole_network_characteristics = calculate_network_characteristics(G)


In [ ]:
layer_char_df = pd.DataFrame(layer_characteristics).transpose()

# Add the whole network characteristics
layer_char_df.loc['Whole Network'] = whole_network_characteristics

# Display the results
print("\nBasic Network Characteristics:")
layer_char_df

In [ ]:

def time_to_seconds(t):
    h, m, s = map(int, t.split(':'))
    return h * 3600 + m * 60 + s

stop_times['arrival_time_sec'] = stop_times['arrival_time'].apply(time_to_seconds)
stop_times['departure_time_sec'] = stop_times['departure_time'].apply(time_to_seconds)

stop_times_trips = pd.merge(stop_times, trips, on='trip_id')
stop_times_trips_routes = pd.merge(stop_times_trips, routes, on='route_id')
stop_times_full = pd.merge(stop_times_trips_routes, stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id')


stop_times_full = stop_times_full.sort_values(['trip_id', 'stop_sequence'])
edge_list = []

for trip_id, trip_data in stop_times_full.groupby('trip_id'):
    trip_data = trip_data.reset_index(drop=True)
    for i in range(len(trip_data) - 1):
        from_stop = trip_data.loc[i]
        to_stop = trip_data.loc[i + 1]
        edge = {
            'from_stop_id': from_stop['stop_id'],
            'to_stop_id': to_stop['stop_id'],
            'start_time': from_stop['departure_time_sec'],
            'end_time': to_stop['arrival_time_sec'],
            'route_type': from_stop['route_type'],
            'trip_id': trip_id
        }
        edge_list.append(edge)

edges_df = pd.DataFrame(edge_list)

stop_positions = stops.set_index('stop_id')[['stop_lat', 'stop_lon']]

fig, ax = plt.subplots(figsize=(12, 8))


# parameters
start_time = 2 * 3600 
end_time = 24 * 3600   
time_step = 10 * 60

time_frames = np.arange(start_time, end_time, time_step)


# Prepare a base graph (optional)
base_G = nx.DiGraph()
base_G.add_nodes_from(stop_positions.index)
pos = {stop_id: (row['stop_lon'], row['stop_lat']) for stop_id, row in stop_positions.iterrows()}





In [49]:
from pyproj import CRS, Transformer
wgs84 = CRS('EPSG:4326')      # WGS84 Latitude/Longitude
utm34n = CRS('EPSG:32634')    # UTM zone 34N
transformer = Transformer.from_crs(wgs84, utm34n, always_xy=True)
# Apply the transformation to stop positions
def transform_coords(row):
    x, y = transformer.transform(row['stop_lon'], row['stop_lat'])
    return pd.Series({'x': x, 'y': y})

stop_positions[['x', 'y']] = stop_positions.apply(transform_coords, axis=1)
pos = {stop_id: (row['x'], row['y']) for stop_id, row in stop_positions.iterrows()}
min_x, max_x = stop_positions['x'].min(), stop_positions['x'].max()
min_y, max_y = stop_positions['y'].min(), stop_positions['y'].max()
padding = 1000

ax.set_xlim(min_x - padding, max_x + padding)
ax.set_ylim(min_y - padding, max_y + padding)

missing_coords = stop_positions[stop_positions[['x', 'y']].isnull().any(axis=1)]
if not missing_coords.empty:
    print("Warning: Some stops have missing coordinates:")
    print(missing_coords)
    stop_positions = stop_positions.dropna(subset=['x', 'y'])



times = []
num_nodes_list = []
num_edges_list = []
avg_degree_list = []
density_list = []
num_components_list = []
avg_clustering_list = []
largest_component_size_list = []

def update(frame_time):


    ax.clear()
    current_time_str = f"{int(frame_time // 3600):02d}:{int((frame_time % 3600) // 60):02d}"
    ax.set_title(f"Traffic Flow at {int(frame_time // 3600):02d}:{int((frame_time % 3600) // 60):02d}")
    ax.axis('off')
    ax.set_aspect('equal')
    
    ax.set_xlim(min_x - padding, max_x + padding)
    ax.set_ylim(min_y - padding, max_y + padding)
    
    active_edges = edges_df[
        (edges_df['start_time'] <= frame_time) &
        (edges_df['end_time'] >= frame_time)
    ]
    
    G = nx.DiGraph()
    G.add_nodes_from(stop_positions.index)
    
    for _, edge in active_edges.iterrows():
        G.add_edge(
            edge['from_stop_id'],
            edge['to_stop_id'],
            route_type=edge['route_type'],
            trip_id=edge['trip_id']
        )
    
    # Compute network metrics
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    avg_degree = (sum(dict(G.degree()).values()) / num_nodes) if num_nodes > 0 else 0
    density = nx.density(G)
    num_components = nx.number_weakly_connected_components(G)
    avg_clustering = nx.average_clustering(G.to_undirected())
    largest_cc = max(nx.weakly_connected_components(G), key=len)
    largest_component_size = len(largest_cc)

    # Store the metrics
    times.append(frame_time)
    num_nodes_list.append(num_nodes)
    num_edges_list.append(num_edges)
    avg_degree_list.append(avg_degree)
    density_list.append(density)
    num_components_list.append(num_components)
    avg_clustering_list.append(avg_clustering)
    largest_component_size_list.append(largest_component_size)
    # avg_shortest_path_length_list.append(aspl)
    textstr = '\n'.join((
        f"Time: {current_time_str}",
        f"Nodes: {num_nodes}",
        f"Edges: {num_edges}",
        f"Avg Degree: {avg_degree:.2f}",
        f"Density: {density:.4f}",
        f"Components: {num_components}",
        f"Largest CC Size: {largest_component_size}",
        f"Avg Clustering: {avg_clustering:.4f}",
        # f"Avg Shortest Path Length: {aspl:.2f}" if aspl else "ASPL: N/A",
    ))
    
    props = dict(boxstyle='round', facecolor='white', alpha=0.7)
    ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=props)

    # Add background map
    ctx.add_basemap(ax, crs=utm34n.to_string(), source=ctx.providers.OpenStreetMap.Mapnik)

    nx.draw_networkx_nodes(G, pos, node_size=1, node_color='black', ax=ax)
    

    for route_type, color in transport_colors.items():
        edges_of_type = [(u, v) for u, v, d in G.edges(data=True) if d['route_type'] == route_type]
        nx.draw_networkx_edges(G, pos, edgelist=edges_of_type, edge_color=color, ax=ax, arrows=False, width=1)
    
    legend_elements = []
    for route_type, color in transport_colors.items():
        legend_elements.append(plt.Line2D([0], [0], color=color, lw=2, label=transport_types[route_type]))
    ax.legend(handles=legend_elements, title='Transport Types', loc='upper right')


In [ ]:
import matplotlib as mpl

mpl.rcParams['animation.embed_limit'] = 100 * 1024 * 1024  # 100 MB in bytes

anim = FuncAnimation(fig, update, frames=time_frames, interval=100, repeat=False)
HTML(anim.to_jshtml())


how far can I get from the city center within an hour

In [ ]:
city_center_stops = stops[stops['stop_name'].str.contains('Centrum', case=False, na=False)]
#print(city_center_stops[['stop_id', 'stop_name']])
city_center_stop_id = city_center_stops.iloc[0]['stop_id']
def time_to_seconds(t):
    h, m, s = map(int, t.split(':'))
    return h * 3600 + m * 60 + s

# Apply conversion
stop_times['arrival_time_sec'] = stop_times['arrival_time'].apply(time_to_seconds)
stop_times['departure_time_sec'] = stop_times['departure_time'].apply(time_to_seconds)

stop_times_trips = pd.merge(stop_times, trips, on='trip_id')
stop_times_trips_routes = pd.merge(stop_times_trips, routes, on='route_id')
stop_times_full = pd.merge(stop_times_trips_routes, stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id')

stop_times_full = stop_times_full.sort_values(['trip_id', 'stop_sequence'])

edge_list = []
for trip_id, trip_data in stop_times_full.groupby('trip_id'):
    trip_data = trip_data.reset_index(drop=True)
    for i in range(len(trip_data) - 1):
        from_stop = trip_data.loc[i]
        to_stop = trip_data.loc[i + 1]
        edge = {
            'from_stop_id': from_stop['stop_id'],
            'to_stop_id': to_stop['stop_id'],
            'departure_time': from_stop['departure_time_sec'],
            'arrival_time': to_stop['arrival_time_sec'],
            'trip_id': trip_id,
            'route_id': from_stop['route_id'],
            'route_type': from_stop['route_type']
        }
        edge_list.append(edge)

edges_df = pd.DataFrame(edge_list)


In [ ]:

G = nx.MultiDiGraph()

for _, stop in stops.iterrows():
    G.add_node(stop['stop_id'], 
               stop_name=stop['stop_name'], 
               stop_lat=stop['stop_lat'], 
               stop_lon=stop['stop_lon'])

for _, edge in edges_df.iterrows():
    G.add_edge(
        edge['from_stop_id'], 
        edge['to_stop_id'], 
        departure_time=edge['departure_time'],
        arrival_time=edge['arrival_time'],
        trip_id=edge['trip_id'],
        route_id=edge['route_id'],
        route_type=edge['route_type']
    )


def time_dependent_dijkstra(G, source, departure_time, max_travel_time=3600):
    earliest_arrival = {node: np.inf for node in G.nodes()}
    earliest_arrival[source] = departure_time
    heap = [(departure_time, source)]
    predecessors = {}

    while heap:
        current_time, u = heapq.heappop(heap)
        if current_time - departure_time > max_travel_time:
            break

        for v, attrs in G[u].items():
            for key, edge_data in attrs.items():
                wait_time = max(0, edge_data['departure_time'] - current_time)
                arrival_time = current_time + wait_time + (edge_data['arrival_time'] - edge_data['departure_time'])

                # If arrival time is better, update
                if arrival_time < earliest_arrival[v]:
                    earliest_arrival[v] = arrival_time
                    heapq.heappush(heap, (arrival_time, v))
                    predecessors[v] = (u, edge_data)
    
    return earliest_arrival, predecessors

departure_time_str = '08:00:00'
departure_time_sec = time_to_seconds(departure_time_str)
max_travel_time = 3600

earliest_arrival_times, predecessors = time_dependent_dijkstra(
    G, 
    source=city_center_stop_id, 
    departure_time=departure_time_sec, 
    max_travel_time=max_travel_time
)
reachable_stops = {stop_id: arrival_time for stop_id, arrival_time in earliest_arrival_times.items()
                   if arrival_time - departure_time_sec <= max_travel_time}

print(f"Number of reachable stops within one hour: {len(reachable_stops)}")

In [ ]:
import matplotlib.pyplot as plt

reachable_stops_df = stops[stops['stop_id'].isin(reachable_stops.keys())]
reachable_stops_df['arrival_time'] = reachable_stops_df['stop_id'].map(reachable_stops)
reachable_stops_df['travel_time'] = reachable_stops_df['arrival_time'] - departure_time_sec
reachable_stops_df['travel_time_minutes'] = reachable_stops_df['travel_time'] / 60.0
plt.figure(figsize=(12, 8))
plt.scatter(stops['stop_lon'], stops['stop_lat'], color='lightgray', s=5, label='All Stops')
plt.scatter(reachable_stops_df['stop_lon'], reachable_stops_df['stop_lat'], 
            c=reachable_stops_df['travel_time_minutes'], cmap='viridis', s=20, label='Reachable Stops')
city_center_stop = stops[stops['stop_id'] == city_center_stop_id]
plt.scatter(city_center_stop['stop_lon'], city_center_stop['stop_lat'], color='red', s=50, label='City Center')

plt.colorbar(label='Travel Time (minutes)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title(f"Reachable Stops within One Hour from City Center at {departure_time_str}")
plt.legend()
plt.show()


In [ ]:
paths = {}
for stop_id in reachable_stops.keys():
    path = []
    current = stop_id
    while current != city_center_stop_id:
        if current in predecessors:
            prev, edge_data = predecessors[current]
            path.append((prev, current))
            current = prev
        else:
            break  # No path found
    paths[stop_id] = path

plt.figure(figsize=(12, 8))
plt.scatter(stops['stop_lon'], stops['stop_lat'], color='lightgray', s=5, label='All Stops')
plt.scatter(reachable_stops_df['stop_lon'], reachable_stops_df['stop_lat'], 
            c=reachable_stops_df['travel_time_minutes'], cmap='viridis', s=20, label='Reachable Stops')
plt.scatter(city_center_stop['stop_lon'], city_center_stop['stop_lat'], color='red', s=50, label='City Center')

for path in paths.values():
    path_coords = [(stops.loc[stops['stop_id'] == u, 'stop_lon'].values[0], 
                    stops.loc[stops['stop_id'] == u, 'stop_lat'].values[0]) for u, v in path]
    path_coords.append((stops.loc[stops['stop_id'] == stop_id, 'stop_lon'].values[0], 
                        stops.loc[stops['stop_id'] == stop_id, 'stop_lat'].values[0]))
    lon, lat = zip(*path_coords)
    plt.plot(lon, lat, color='blue', alpha=0.5)

plt.colorbar(label='Travel Time (minutes)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title(f"Reachable Stops and Paths within One Hour from City Center at {departure_time_str}")
plt.legend()
plt.show()
